In [4]:
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver saves the state into RAM. On closing the app, state will be erased.
# This is used for testing purpose. It is mostly not used in production.

In [5]:
load_dotenv()

True

In [8]:
model_gemini = ChatGoogleGenerativeAI(
    model = 'gemini-2.5-flash',
    temperature = 0.9
)

In [9]:
# Defining the state

class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [41]:
# Creating nodes

def generate_joke(state: JokeState) ->JokeState:

    topic = state['topic']

    prompt = f'''Generate a wise and teachink type joke on {topic}.'''

    response = model_gemini.invoke(prompt).content

    return {"joke": response}

#---------------------------------------------------------------------------------------

def explain_joke(state: JokeState) ->JokeState:

    topic = state['topic']
    joke = state['joke']
    

    prompt = f'''Explain the below given joke on {topic}, in two lines
    
    {joke}'''

    response = model_gemini.invoke(prompt).content

    return {'explanation': response}

In [42]:
# Creating graph
graph = StateGraph(JokeState)

# Adding nodes
graph.add_node('generate_joke', generate_joke)
graph.add_node('explain_joke', explain_joke)

# Creating edges
graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'explain_joke')
graph.add_edge('explain_joke', END)

# Creating a checkpointer's object
checkpointer = InMemorySaver()

# Compiling the graph with checkpointer
workflow = graph.compile(checkpointer = checkpointer)

In [43]:
config1 = {"configurable": {"thread_id": '1'}}

initial_state = {'topic': 'Sky'}

# invoking the workflow
response = workflow.invoke(input = initial_state, config = config1)

In [44]:
print(response['explanation'])

The joke uses the literal sky as a metaphor for Sky TV. It suggests Sky's initial offers appear simple and appealing (like a clear blue sky), but there's a universe of hidden complexity and costs.

The punchline implies that the "real picture" of Sky's true details and charges (the "stars") only becomes clear later, "when the sun goes down" (e.g., when the bill arrives or you deal with customer service).


In [46]:
# Getting the saved state
print(workflow.get_state(config1))

StateSnapshot(values={'topic': 'Sky', 'joke': 'A wise old sage was asked by his eager student, "Master, what is the most profound lesson the sky teaches us?"\n\nThe Master gazed upwards thoughtfully. "Ah, young one, the sky teaches us a fundamental truth about perception and reality."\n\n"Is it about its vastness, or the ever-changing clouds?" the student inquired.\n\n"Those are mere footnotes," the Master chuckled. "Its deepest wisdom is this: It shows us that even when something appears entirely blue, clear, and utterly simple, there is an entire universe of profound complexity, ancient stories, and countless sparkling truths *hidden just behind that cheerful facade*."\n\n"So, it teaches us not to judge by appearances, and to always look deeper for what\'s truly there?" the student concluded.\n\n"Precisely!" the Master smiled. "And also, that if you truly want to see the *real* picture... sometimes you just have to wait until the sun goes down."', 'explanation': 'The joke uses the li

In [47]:
print(workflow.get_state_history(config1))

<generator object Pregel.get_state_history at 0x000001F7E8DBC880>


In [49]:
l = list(workflow.get_state_history(config1))

for history in l:
    print(history)
    print()
    print()

StateSnapshot(values={'topic': 'Sky', 'joke': 'A wise old sage was asked by his eager student, "Master, what is the most profound lesson the sky teaches us?"\n\nThe Master gazed upwards thoughtfully. "Ah, young one, the sky teaches us a fundamental truth about perception and reality."\n\n"Is it about its vastness, or the ever-changing clouds?" the student inquired.\n\n"Those are mere footnotes," the Master chuckled. "Its deepest wisdom is this: It shows us that even when something appears entirely blue, clear, and utterly simple, there is an entire universe of profound complexity, ancient stories, and countless sparkling truths *hidden just behind that cheerful facade*."\n\n"So, it teaches us not to judge by appearances, and to always look deeper for what\'s truly there?" the student concluded.\n\n"Precisely!" the Master smiled. "And also, that if you truly want to see the *real* picture... sometimes you just have to wait until the sun goes down."', 'explanation': 'The joke uses the li

# Note: Langchain returns the history in reverse order. The last one is shown first and so on.

* Langgraph stores two keys, values and result
* 'task' shows previous keys and values shows what that particular node performed.
* Here (from last),

* `Step: -1:` value = {}, bcz it is the START node, result{topic: Sky}, means this node initialized topic = Sky
* `Step 0:` value = {topic: Sky} Value initialized by previous node, result{joke: '...'}, this node generated a joke
* `Step 1:` value = {topic: Sky, joke: '...'}, task{'explanation': '...'}, This node generated explanation
* `Step 2:` value = {topic: Sky, joke: '...', explanation: '...'}, result = {} bcz its is the END node 

# Last step returns the complete state

# General Info:

* We can change thread_id and both ids will get stored in RAM and we can get state of both IDs.
* Also, upon closing of this program, all the state will be erased as we are storing it in RAM.
* Had we stored it in any database, even after 5 days, we can retrieve state of all the threads.

# Next notebook: Practical of `Fault Tolerance`

`Fault tolerance` means if the execution of the workflow gets stopped in between `(due to any reason)`, after solving the issue, we can resume the workflow from where it stopped.